# 03 — Attention: scaled dot-product, masking, multi-head

**Paper:** Vaswani et al. (2017), *Attention Is All You Need*, §3.2 (Eq. 1, footnote 4, and the multi-head equations in §3.2.2).

**You will learn**
- how to read Eq. 1 with the batch and head dims the paper leaves out
- masks: additive $-\infty$ vs. boolean, and why fully masked rows produce `nan`
- the split-heads / merge-heads reshape that every transformer uses
- *why* the $\frac{1}{\sqrt{d_k}}$ is there, verified empirically from footnote 4

**Rule:** don't use `F.scaled_dot_product_attention` or `nn.MultiheadAttention` in your solutions. You may use `torch.softmax` now that you've built it in notebook 01.

In [ ]:
import math
import torch
import torch.nn as nn
import torch.nn.functional as F
import matplotlib.pyplot as plt
from p2t import check, check_grad, seed

seed(0)

## 1. Scaled dot-product attention

$$\mathrm{Attention}(Q, K, V) = \mathrm{softmax}\!\left(\frac{QK^\top}{\sqrt{d_k}}\right)V$$

**Decode it** (step 2 of the recipe, writing down shapes):

| symbol | paper shape | code shape |
|---|---|---|
| $Q$ | $(n_q, d_k)$ | `(B, H, Tq, dk)` |
| $K$ | $(n_k, d_k)$ | `(B, H, Tk, dk)` |
| $V$ | $(n_k, d_v)$ | `(B, H, Tk, dv)` |
| $QK^\top$ | $(n_q, n_k)$ | `(B, H, Tq, Tk)` |
| output | $(n_q, d_v)$ | `(B, H, Tq, dv)` |

Softmax over **which axis**? Each query needs a distribution over keys, so the reduction is over `Tk`, the last dim.

**Masking:** the paper sets illegal positions to $-\infty$ before the softmax (§3.2.3), so they get probability $e^{-\infty} = 0$. Here, `mask` is boolean with **`True` = allowed to attend** (the same convention as `F.scaled_dot_product_attention`). Use `masked_fill`.

### Exercise 1 — `sdpa`

In [ ]:
def sdpa(q, k, v, mask=None):
    """q: (..., Tq, dk), k: (..., Tk, dk), v: (..., Tk, dv), mask: bool broadcastable to (..., Tq, Tk)
    returns (..., Tq, dv)"""
    # YOUR CODE HERE
    raise NotImplementedError

In [ ]:
B, H, Tq, Tk, dk, dv = 2, 4, 5, 7, 16, 8
q, k, v = torch.randn(B, H, Tq, dk), torch.randn(B, H, Tk, dk), torch.randn(B, H, Tk, dv)
check("sdpa", sdpa(q, k, v), F.scaled_dot_product_attention(q, k, v))
check_grad("sdpa", sdpa, F.scaled_dot_product_attention, q, k, v)

mask = torch.rand(Tq, Tk) > 0.3
mask[:, 0] = True  # guarantee every query can see at least one key
check("sdpa + mask", sdpa(q, k, v, mask), F.scaled_dot_product_attention(q, k, v, attn_mask=mask))

### Exercise 2 — causal mask

For autoregressive decoding, position $i$ may only attend to positions $j \le i$. Build the `(T, T)` boolean mask. (Hint: which triangle is `True`?)

In [ ]:
def causal_mask(T):
    # YOUR CODE HERE
    raise NotImplementedError

In [ ]:
print(causal_mask(4).int())
q, k, v = torch.randn(B, H, 6, dk), torch.randn(B, H, 6, dk), torch.randn(B, H, 6, dv)
check("causal sdpa", sdpa(q, k, v, causal_mask(6)), F.scaled_dot_product_attention(q, k, v, is_causal=True))

**Try it:** make a mask with one row that is entirely `False` and run `sdpa`. You get `nan`, because softmax of all $-\infty$ is $0/0$. This happens in practice with left-padded batches. Real implementations either guarantee each row has at least one `True` or replace $-\infty$ with a large finite negative number.

## 2. Multi-head attention

§3.2.2:
$$\mathrm{MultiHead}(Q,K,V) = \mathrm{Concat}(\mathrm{head}_1,\dots,\mathrm{head}_h)\,W^O, \qquad \mathrm{head}_i = \mathrm{Attention}(QW_i^Q, KW_i^K, VW_i^V)$$
with $W_i^Q \in \mathbb{R}^{d_{model}\times d_k}$ and $d_k = d_{model}/h$.

**The implementation trick:** $h$ separate $d_{model}\times d_k$ matrices stacked side by side form one $d_{model}\times d_{model}$ matrix. So you do **one** big projection and then *reshape* to expose the heads:
```
(B, T, D) --proj--> (B, T, D) --view--> (B, T, H, dh) --transpose--> (B, H, T, dh)
```
After attention, you reverse this ("merge heads") and apply $W^O$.

Weights are in `nn.Linear` layout `(out, in)`, so projecting a batch `x` is `x @ W.T` (see notebook 00).

### Exercise 3 — self-attention with multiple heads

In [ ]:
def split_heads(x, n_heads):
    """(B, T, D) -> (B, H, T, D/H)"""
    # YOUR CODE HERE
    raise NotImplementedError


def merge_heads(x):
    """(B, H, T, dh) -> (B, T, H*dh)"""
    # YOUR CODE HERE
    raise NotImplementedError


def multi_head_self_attention(x, Wq, Wk, Wv, Wo, n_heads, causal=False):
    """x: (B, T, D); W*: (D, D) in nn.Linear layout. Returns (B, T, D)."""
    # YOUR CODE HERE
    raise NotImplementedError

In [ ]:
B, T, D, Hh = 2, 6, 32, 4
x = torch.randn(B, T, D)
Wq, Wk, Wv, Wo = (torch.randn(D, D) / math.sqrt(D) for _ in range(4))

x_split = split_heads(x, Hh)
check("split/merge round-trip", merge_heads(x_split), x)
check("split_heads puts head h's features at [h*dh:(h+1)*dh]", x_split[:, 1], x[:, :, 8:16])

ref = nn.MultiheadAttention(D, Hh, bias=False, batch_first=True)
with torch.no_grad():
    ref.in_proj_weight.copy_(torch.cat([Wq, Wk, Wv]))
    ref.out_proj.weight.copy_(Wo)
    expected, _ = ref(x, x, x, need_weights=False)
    # NOTE: nn.MultiheadAttention's bool attn_mask uses the OPPOSITE convention (True = blocked)!
    expected_causal, _ = ref(x, x, x, need_weights=False, attn_mask=~causal_mask(T))
check("multi-head attention", multi_head_self_attention(x, Wq, Wk, Wv, Wo, Hh), expected)
check("multi-head attention (causal)", multi_head_self_attention(x, Wq, Wk, Wv, Wo, Hh, causal=True), expected_causal)

The `NOTE` in that test cell matters: two PyTorch APIs use **opposite** boolean-mask conventions. Whenever you meet a mask in code or in a paper, check which value means "keep".

## 3. Experiment — why divide by $\sqrt{d_k}$?

Footnote 4 of the paper:
> *To illustrate why the dot products get large, assume that the components of $q$ and $k$ are independent random variables with mean 0 and variance 1. Then their dot product, $q\cdot k = \sum_{i=1}^{d_k} q_i k_i$, has mean 0 and variance $d_k$.*

You can check this claim numerically, and then check its consequence: large logits make softmax nearly one-hot (**saturated**), and a saturated softmax has near-zero gradients.

### Exercise 4 — verify footnote 4
Return the empirical variance of $q \cdot k$ for random unit-variance vectors, **without** and **with** the scaling.

In [ ]:
def dot_product_variance(dk, n=100_000):
    """Return (var of q.k, var of q.k/sqrt(dk)) for q, k ~ N(0, I_dk)."""
    # YOUR CODE HERE
    raise NotImplementedError

In [ ]:
seed(0)
for dk in [16, 64, 256]:
    raw, scaled = dot_product_variance(dk)
    print(f"dk={dk:4d}  var(q·k)={raw:8.2f}   var(q·k/√dk)={scaled:.3f}")
    check(f"  var ≈ dk", raw, torch.tensor(float(dk)), rtol=0.05)
    check(f"  scaled var ≈ 1", scaled, torch.tensor(1.0), rtol=0.05)

In [ ]:
# Consequence: softmax saturation and gradient size, with vs. without scaling.
seed(0)
dks = [4, 16, 64, 256, 1024]
for scaled in [False, True]:
    ents, grads = [], []
    for dk in dks:
        q = torch.randn(1, dk); k = torch.randn(32, dk)
        s = (q @ k.T) / (math.sqrt(dk) if scaled else 1.0)
        s.requires_grad_()
        p = torch.softmax(s, -1)
        ents.append(-(p * p.clamp_min(1e-30).log()).sum().item())
        p[0, 0].backward()
        grads.append(s.grad.abs().mean().item())
    plt.subplot(1, 2, 1); plt.semilogx(dks, ents, "o-", label=f"scaled={scaled}")
    plt.subplot(1, 2, 2); plt.loglog(dks, grads, "o-", label=f"scaled={scaled}")
plt.subplot(1, 2, 1); plt.xlabel("d_k"); plt.ylabel("attention entropy"); plt.legend()
plt.subplot(1, 2, 2); plt.xlabel("d_k"); plt.ylabel("mean |grad| wrt scores"); plt.legend()
plt.tight_layout(); plt.show()

## Reflection
1. What's the time and memory complexity of `sdpa` in `T`? Which tensor causes it?
2. In cross-attention (decoder attending to encoder), which of $Q, K, V$ come from which sequence? Does your `sdpa` already support it?
3. Multi-query attention (Shazeer, 2019) shares one $K$ and one $V$ head across all query heads. Which shapes change, and does your `sdpa` still work thanks to broadcasting?